# **Checkpoint 2: Model Training for Fake Review Detection**

###**Import Libraries**

In [26]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder


### **Load and Preparation the Dataset**

In [27]:
# Load the preprocessed dataset
data = pd.read_csv('/content/preprocessed_data.csv')
        # Replace with the column containing labels


In [28]:
data.head(5)

,category,rating,label,text_,cleaned_text,vectorized
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor...",love well made sturdy comfortable love itvery ...,[0. 0. 0. 0. 0...
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I...",love great upgrade original mine couple year,[0. 0. 0. 0. 0...
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...,pillow saved back love look feel pillow,[0. 0. 0. 0. 0...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i...",missing information use great product price,[0. 0. 0. 0. 0...
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...,nice set good quality set two month not,[0. 0. 0. 0. 0...


In [29]:
print(data['cleaned_text'].isna().sum())


60


In [37]:
data = data.dropna(subset=['cleaned_text'])


In [38]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(data['label'])


In [39]:
X = data['cleaned_text']


In [40]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


### **Creating Pipeline**

In [41]:
# Initialize classifiers
classifiers = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000)
}


In [48]:
# Initialize a dictionary to store evaluation results
evaluation_results = {}

# Train and evaluate each classifier with progress bar
for name, classifier in tqdm(classifiers.items(), desc="Training Models", unit="model"):
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=1000)),
        ('classifier', classifier)
    ])

    # Add progress bar for model training
    with tqdm(total=len(X_train), desc=f"Training {name}", unit="sample") as pbar:
        # Train the model (iterating through training data for progress update)
        pipeline.fit(X_train, y_train)
        pbar.update(len(X_train))  # Update progress bar after fitting the model

    # Save the trained model
    model_filename = f'{name.lower().replace(" ", "_")}_model.pkl'
    joblib.dump(pipeline, model_filename)

    # Evaluate the model
    y_pred = pipeline.predict(X_test)
    evaluation_results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='binary'),
        "Recall": recall_score(y_test, y_pred, average='binary'),
        "F1 Score": f1_score(y_test, y_pred, average='binary')
    }
    # Display evaluation results for each model immediately after evaluation
    print(f"\nModel: {name}")
    print(f"  Accuracy: {evaluation_results[name]['Accuracy']:.4f}")
    print(f"  Precision: {evaluation_results[name]['Precision']:.4f}")
    print(f"  Recall: {evaluation_results[name]['Recall']:.4f}")
    print(f"  F1 Score: {evaluation_results[name]['F1 Score']:.4f}")
    print("-" * 40)



Training Models:  33%|███▎      | 1/3 [00:55<01:51, 55.66s/model]


Model: Random Forest
  Accuracy: 0.8326
  Precision: 0.8553
  Recall: 0.8078
  F1 Score: 0.8309
----------------------------------------



Training Models:  67%|██████▋   | 2/3 [21:11<12:18, 738.15s/model]


Model: SVM
  Accuracy: 0.8670
  Precision: 0.8665
  Recall: 0.8733
  F1 Score: 0.8699
----------------------------------------



Training Models: 100%|██████████| 3/3 [21:13<00:00, 424.39s/model]


Model: Logistic Regression
  Accuracy: 0.8409
  Precision: 0.8468
  Recall: 0.8392
  F1 Score: 0.8430
----------------------------------------


In [49]:
# Display evaluation results
for model, metrics in evaluation_results.items():
    print(f"Model: {model}")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")
    print("-" * 30)


Model: Random Forest
  Accuracy: 0.8326
  Precision: 0.8553
  Recall: 0.8078
  F1 Score: 0.8309
------------------------------
Model: SVM
  Accuracy: 0.8670
  Precision: 0.8665
  Recall: 0.8733
  F1 Score: 0.8699
------------------------------
Model: Logistic Regression
  Accuracy: 0.8409
  Precision: 0.8468
  Recall: 0.8392
  F1 Score: 0.8430
------------------------------
